In [ ]:
from urllib.parse import urlparse
import requests
import yaml
from pathlib import Path

root = Path.cwd()

# Constants from yml files
with open(root / "conf" / "config.yml", 'r') as f:
    config = yaml.safe_load(f)

with open(root / "conf" / "symbol_count.yml", 'r') as g:
    symbol_count = yaml.safe_load(g)

headers = config['headers']

session = requests.Session()

class URLFetch:

    def __init__(self, url, method='get', json=False, session=None,
                 headers = None, proxy = None):
        self.url = url
        self.method = method
        self.json = json

        if not session:
            self.session = requests.Session()
        else:
            self.session = session

        if headers:
            self.session.headers.update(headers)
        if proxy:
            self.update_proxy(proxy)
        else:
            self.update_proxy('')

    def set_session(self, session):
        self.session = session
        return self

    def get_session(self, session):
        self.session = session
        return self

    def __call__(self, *args, **kwargs):
        u = urlparse(self.url)
        self.session.headers.update({'Host': u.hostname})
        url = self.url%(args)
        if self.method == 'get':
            return self.session.get(url, params=kwargs, proxies = self.proxy )
        elif self.method == 'post':
            if self.json:
                return self.session.post(url, json=kwargs, proxies = self.proxy )
            else:
                return self.session.post(url, data=kwargs, proxies = self.proxy )

    def update_proxy(self, proxy):
        self.proxy = proxy
        self.session.proxies.update(self.proxy)

    def update_headers(self, headers):
        self.session.headers.update(headers)



In [ ]:
from functools import partial
from helper import nse_symbol_purify

URLFetchSession = partial(URLFetch, session=session,
                          headers=headers)

symbol_count_url = URLFetchSession(
    url='http://www1.nseindia.com/marketinfo/sym_map/symbolCount.jsp')

def get_symbol_count(symbol):

    symbol = nse_symbol_purify(symbol)

    try:
        return symbol_count[symbol]
    except:
        cnt = symbol_count_url(symbol=symbol).text.lstrip().rstrip()
        symbol_count[symbol] = cnt
        return cnt

In [ ]:
symbol_count_url(symbol='BAGFILMS').text

In [ ]:
import datetime

end = datetime.datetime.now().date()
start = end-datetime.timedelta(days=130)

params = {}
params['symbol'] = 'SBIN'
params['fromDate'] = start.strftime('%d-%m-%Y')
params['toDate'] = end.strftime('%d-%m-%Y')
params['series'] = 'EQ'

url = 'http://www1.nseindia.com/products/dynaContent/common/productsSymbolMapping.jsp'

In [ ]:
URLFetchSession(url='http://www1.nseindia.com/marketinfo/sym_map/symbolCount.jsp')

In [ ]:
equity_history_url_full = URLFetchSession(
    url=url)

equity_history_url = partial(equity_history_url_full,
                             dataType='PRICEVOLUMEDELIVERABLE',
                             segmentLink=3, dateRange="")

In [ ]:
equity_history_url

In [ ]:
# Trying my own history!!

from pathlib import Path
from bs4 import BeautifulSoup
import pandas as pd
from functools import partial
import datetime

import requests
import yaml

symbol = 'SBIN'

# Constants from yml files
root = Path().cwd() 

with open(root / "conf" / "config.yml", 'r') as f:
    config = yaml.safe_load(f)


headers = config['headers']
params = config['eq_history_params']

url = 'http://www1.nseindia.com/products/dynaContent/common/productsSymbolMapping.jsp'
# params={'dataType': 'PRICEVOLUMEDELIVERABLE', 'segmentLink': 3, 'dateRange': '', 
#         'symbol': 'INFY', 'series': 'EQ', 'symbolCount': '2', 'fromDate': '28-12-2021', 'toDate': '07-05-2022'}

end = datetime.datetime.now().date()
start = end-datetime.timedelta(days=130)

params['symbol'] = 'SBIN'
params['fromDate'] = start.strftime('%d-%m-%Y')
params['toDate'] = end.strftime('%d-%m-%Y')
params['series'] = 'EQ'
params['symbolCount'] = '2'

session = requests.Session()
resp = session.get(url=url, params=params, headers=headers)
soup = BeautifulSoup(resp.text, 'lxml')
table = soup.find_all('table')
pd.read_html(str(table))[0]


In [21]:
>>> from nsepy import get_history
>>> from datetime import date
>>> data = get_history(symbol="NIFTY", start=date(2015,1,1), end=date(2015,1,31),index=True)

In [22]:
data

,Open,High,Low,Close,Volume,Turnover
Date,,,,,,
2015-01-01,8272.80,8294.70,8248.75,8284.00,56560411,2.321880e+10
2015-01-02,8288.70,8410.60,8288.70,8395.45,101887024,4.715720e+10
2015-01-05,8407.95,8445.60,8363.90,8378.40,118160545,5.525520e+10
2015-01-06,8325.30,8327.85,8111.35,8127.35,172799618,8.089190e+10
2015-01-07,8118.65,8151.20,8065.45,8102.10,164075424,7.464330e+10
2015-01-08,8191.40,8243.50,8167.30,8234.60,143802802,8.147400e+10
2015-01-09,8285.45,8303.30,8190.80,8284.50,152612528,9.305950e+10
2015-01-12,8291.35,8332.60,8245.60,8323.00,103153908,5.437760e+10
2015-01-13,8346.15,8356.65,8267.90,8299.40,129561892,6.499200e+10


In [11]:
from nsepy import symbols

In [19]:
df = symbols.get_symbol_list()
df[df.SYMBOL.str.startswith('NI')]

,SYMBOL,NAME OF COMPANY,SERIES,DATE OF LISTING,PAID UP VALUE,MARKET LOT,ISIN NUMBER,FACE VALUE
1146,NIACL,The New India Assurance Company Limited,EQ,13-NOV-2017,5,1,INE470Y01017,5
1147,NIBL,NRB Industrial Bearings Limited,EQ,09-APR-2013,2,1,INE047O01014,2
1148,NIITLTD,NIIT Limited,EQ,16-AUG-2004,2,1,INE161A01038,2
1149,NILAINFRA,Nila Infrastructures Limited,EQ,21-MAY-2015,1,1,INE937C01029,1
1150,NILASPACES,Nila Spaces Limited,EQ,28-DEC-2018,1,1,INE00S901012,1
1151,NILKAMAL,Nilkamal Limited,EQ,01-NOV-1995,10,1,INE310A01015,10
1152,NIPPOBATRY,Indo-National Limited,EQ,06-OCT-1999,5,1,INE567A01028,5
1153,NIRAJ,Niraj Cement Structurals Limited,EQ,01-OCT-2020,10,1,INE368I01016,10
1154,NIRAJISPAT,Niraj Ispat Industries Limited,BE,27-OCT-2016,10,1,INE326T01011,10
1155,NITCO,Nitco Limited,EQ,21-MAR-2006,10,1,INE858F01012,10
